# Agentic REPL model staging: download + publish

Downloads `Qwen3-Coder-30B-A3B-Instruct-Q4_K_M.gguf` from `unsloth/Qwen3-Coder-30B-A3B-Instruct-GGUF` on Hugging Face and publishes it as the Kaggle Dataset `ankitdash24/qwen3-coder-30b-a3b-instruct-gguf`. CPU-only, internet-enabled, no GPU quota consumed. One-time staging step -- see agentic_repl/models/README.md.

## 1. Authenticate the Kaggle CLI from the staged token dataset

In [ ]:
from pathlib import Path
import shutil
import subprocess
import sys

INPUT_ROOT = Path('/kaggle/input')
print('/kaggle/input contents:')
for entry in sorted(INPUT_ROOT.rglob('*')):
    print(' ', entry)

def find_input_file(dataset_slug, filename):
    # Confirmed directly: private dataset inputs can mount at either
    # /kaggle/input/<slug>/<file> or the nested
    # /kaggle/input/datasets/<owner>/<slug>/<file> -- check both rather
    # than assume either.
    flat = INPUT_ROOT / dataset_slug / filename
    if flat.is_file():
        return flat
    nested = sorted(INPUT_ROOT.glob(f'datasets/*/{dataset_slug}/{filename}'))
    if nested:
        return nested[0]
    return None

CREDENTIALS_SRC = find_input_file('agentic-repl-kaggle-token', 'kaggle.json')
if CREDENTIALS_SRC is None:
    raise SystemExit(
        'No kaggle.json found under /kaggle/input -- the credentials dataset '
        'is not mounted. Check dataset_sources in kernel-metadata.json and that '
        'the dataset is attached to this kernel.'
    )
print('Using Kaggle credentials from:', CREDENTIALS_SRC)

KAGGLE_CONFIG_DIR = Path.home() / '.kaggle'
KAGGLE_CONFIG_DIR.mkdir(parents=True, exist_ok=True)
shutil.copy(CREDENTIALS_SRC, KAGGLE_CONFIG_DIR / 'kaggle.json')
(KAGGLE_CONFIG_DIR / 'kaggle.json').chmod(0o600)
print('Wrote Kaggle credentials to', KAGGLE_CONFIG_DIR / 'kaggle.json')

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'kaggle'], check=True)
print('kaggle CLI ready')

HF_TOKEN_SRC = find_input_file('agentic-repl-hf-token', 'hf_token.txt')
HF_TOKEN = HF_TOKEN_SRC.read_text(encoding='utf-8').strip() if HF_TOKEN_SRC else None
if HF_TOKEN:
    print('Found HF token -- downloads will be authenticated (higher rate limit).')
else:
    print('No HF token staged -- downloading anonymously (may be rate-limited/slow).')


## 2. Check free disk space before downloading

`/kaggle/working` is capped around ~21GB total, too tight for an ~18.6GB file plus margin (confirmed directly: 20.9GB free on a fresh session). Download to `/kaggle/tmp` instead, which reports a larger ephemeral quota.

In [ ]:
WORKDIR = Path('/kaggle/working')
DOWNLOAD_DIR = Path('/kaggle/tmp/model')
DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)
REQUIRED_BYTES = int('Qwen3-Coder-30B-A3B-Instruct-Q4_K_M.gguf' and 18.6e9 * 1.15)  # +15% safety margin
total, used, free = shutil.disk_usage(DOWNLOAD_DIR)
print(f'/kaggle/tmp: total={total/1e9:.1f}GB used={used/1e9:.1f}GB free={free/1e9:.1f}GB')
print(f'required (with margin) = {REQUIRED_BYTES/1e9:.1f}GB')

if free < REQUIRED_BYTES:
    raise SystemExit(
        f'Not enough free disk in /kaggle/tmp ({free/1e9:.1f}GB free, '
        f'need ~{REQUIRED_BYTES/1e9:.1f}GB) -- aborting before a partial download. '
        'See agentic_repl/models/README.md for a smaller-model fallback.'
    )


## 3. Download the GGUF from Hugging Face (resumable, size-verified)

Uses HF_TOKEN (if staged) via an Authorization header -- anonymous requests are explicitly rate-limited by Hugging Face (`X-HF-Warning: unauthenticated ... enable higher rate limits and faster downloads`, confirmed directly against this exact URL).

**Resumable by design, not just retried**: the redirect this URL resolves to is a presigned S3-style link valid for exactly `X-Amz-Expires=3600` seconds (confirmed directly in the response headers) -- a first version of this cell hit that expiry mid-transfer (achieved throughput was only ~5MB/s, so the full 18.6GB transfer took longer than the URL's 1-hour validity), the connection closed early, and the code published the truncated ~17.5GB result anyway since nothing checked the final size against Content-Length. Fixed by re-resolving a fresh presigned URL (via a Range request against the *original* HF URL, not the expired redirect) on every retry, and by hard-failing before publishing if the final size doesn't match.

In [ ]:
import time
import urllib.error
import urllib.request

MODEL_URL = 'https://huggingface.co/unsloth/Qwen3-Coder-30B-A3B-Instruct-GGUF/resolve/main/Qwen3-Coder-30B-A3B-Instruct-Q4_K_M.gguf'
MODEL_FILE = 'Qwen3-Coder-30B-A3B-Instruct-Q4_K_M.gguf'
MODEL_DIR = DOWNLOAD_DIR
destination = MODEL_DIR / MODEL_FILE
CHUNK_SIZE = 8 * 1024 * 1024
MAX_ATTEMPTS = 10
SOCKET_TIMEOUT_S = 60

auth_headers = {'Authorization': f'Bearer {HF_TOKEN}'} if HF_TOKEN else {}

expected_size = None
started = time.perf_counter()
for attempt in range(1, MAX_ATTEMPTS + 1):
    existing = destination.stat().st_size if destination.exists() else 0
    headers = dict(auth_headers)
    if existing:
        headers['Range'] = f'bytes={existing}-'
    request = urllib.request.Request(MODEL_URL, headers=headers)
    try:
        with urllib.request.urlopen(request, timeout=SOCKET_TIMEOUT_S) as response:
            if existing and response.status == 206:
                mode, base = 'ab', existing
                content_range_total = response.headers.get('Content-Range', '').rsplit('/', 1)[-1]
                expected_size = int(content_range_total) if content_range_total.isdigit() else expected_size
            else:
                mode, base = 'wb', 0  # server ignored/rejected Range -- restart clean
                expected_size = int(response.headers.get('Content-Length', 0)) or expected_size
            print(f'attempt {attempt}: status={response.status} mode={mode} resume_from={base} expected_size={expected_size}')
            downloaded = base
            next_report_at = downloaded
            with destination.open(mode) as out_file:
                while True:
                    chunk = response.read(CHUNK_SIZE)
                    if not chunk:
                        break
                    out_file.write(chunk)
                    downloaded += len(chunk)
                    if downloaded >= next_report_at:
                        elapsed_so_far = time.perf_counter() - started
                        rate_mb_s = (downloaded / 1e6) / elapsed_so_far if elapsed_so_far > 0 else 0
                        pct = 100 * downloaded / expected_size if expected_size else 0
                        print(f'{downloaded/1e9:.2f}/{(expected_size or 0)/1e9:.2f} GB ({pct:.1f}%) -- {rate_mb_s:.1f} MB/s avg')
                        next_report_at = downloaded + 500 * 1024 * 1024  # every ~500MB
    except (urllib.error.URLError, TimeoutError, ConnectionError, OSError) as exc:
        print(f'attempt {attempt} failed: {exc!r} -- will retry, resuming from current file size')
        continue
    final_size = destination.stat().st_size
    print(f'attempt {attempt} ended with {final_size/1e9:.2f} GB on disk')
    if expected_size and final_size >= expected_size:
        break
else:
    raise SystemExit(
        f'Download did not complete after {MAX_ATTEMPTS} attempts '
        f'({destination.stat().st_size if destination.exists() else 0} bytes on disk, '
        f'expected {expected_size}). Aborting before publishing an incomplete file.'
    )

elapsed = time.perf_counter() - started
final_size = destination.stat().st_size
if expected_size and final_size != expected_size:
    raise SystemExit(
        f'Downloaded size {final_size} != expected {expected_size} -- '
        'aborting before publishing a corrupt/truncated file.'
    )
size_gb = final_size / 1e9
print(f'Downloaded {destination} ({size_gb:.2f} GB, verified against Content-Length) '
      f'in {elapsed:.0f}s ({(size_gb*1000)/elapsed:.1f} MB/s average)')


## 4. Verify checksum

In [ ]:
import hashlib

digest = hashlib.sha256()
with destination.open('rb') as handle:
    for chunk in iter(lambda: handle.read(1 << 20), b''):
        digest.update(chunk)
print('sha256:', digest.hexdigest())


## 5. Publish as a Kaggle Dataset

In [ ]:
import json as json_module

TARGET_DATASET_ID = 'ankitdash24/qwen3-coder-30b-a3b-instruct-gguf'
TARGET_DATASET_TITLE = 'qwen3-coder-30b-a3b-instruct-gguf'
metadata = {
    'title': TARGET_DATASET_TITLE,
    'id': TARGET_DATASET_ID,
    'licenses': [{'name': 'unknown'}],
}
(MODEL_DIR / 'dataset-metadata.json').write_text(json_module.dumps(metadata, indent=2))

check = subprocess.run(
    ['kaggle', 'datasets', 'status', TARGET_DATASET_ID],
    capture_output=True, text=True,
)
dataset_exists = check.returncode == 0
print('dataset_exists =', dataset_exists)
if dataset_exists:
    result = subprocess.run(
        ['kaggle', 'datasets', 'version', '-p', str(MODEL_DIR), '-m', 'update', '-r', 'zip'],
        capture_output=True, text=True,
    )
else:
    result = subprocess.run(
        ['kaggle', 'datasets', 'create', '-p', str(MODEL_DIR), '-r', 'zip'],
        capture_output=True, text=True,
    )
print('--- stdout ---')
print(result.stdout)
print('--- stderr ---')
print(result.stderr)
result.check_returncode()
print('Published dataset:', TARGET_DATASET_ID)
